# AF2 spectral — Stage 1 sequential Kaggle
Menjalankan `AF2WIN → AF2ORI → AF2POL → AF2SOFT → AF2LUM` satu per satu pada satu GPU Kaggle. Notebook ini **Kaggle-only**. Input wajib memakai bundle private yang dibuat oleh `Faruq_V3_AF2_Spectral_Kaggle_Bundle_Colab.ipynb`.
Static audit memisahkan Stage-1 dari PCG1/WAV1. Sebelum training, frontend Stage-1 juga menjalani deterministic CUDA forward/backward smoke-test. Jika subprocess training gagal, traceback utama memuat log tail sehingga root cause tidak tersembunyi oleh `returncode=1`.


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
manifests=sorted(p for p in INPUT.rglob('af2_spectral_kaggle_manifest.json') if p.is_file())
d0_named=sorted(p for p in INPUT.rglob('D0_seed42_best.pt') if p.is_file())
if len(manifests)!=1 or len(d0_named)!=1: raise FileNotFoundError(f'STOP CEPAT: manifest={manifests}, D0={d0_named}')
print('FAST INPUT PREFLIGHT PASS'); print('MANIFEST:',manifests[0]); print('D0 NAMED:',d0_named[0])


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile,torch
os.chdir(WORK); REPO=WORK/'coffee-bean-detection'; BRANCH='agent/af2-spectral-factorization'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input,restore_spectral_kaggle_run
from coffee_detector.af2_spectral.audit import run_spectral_static_audit,STAGE1_ARMS
from coffee_detector.af2_spectral import SpectralInputEnhancer
from coffee_detector.af2_spectral.config import frozen_arm_config
DATA,ARTIFACTS,INPUT_CONTRACT=prepare_af2_spectral_kaggle_input(INPUT,WORK)
assert INPUT_CONTRACT['decision']=='PASS' and INPUT_CONTRACT['test_images_accessed'] is False
D0=ARTIFACTS['D0_seed42_best.pt']; print('FULL INPUT CONTRACT PASS'); print('DATA:',DATA); print('D0:',D0)
OUT=WORK/'af2-spectral-factorization-v1'; OUT.mkdir(exist_ok=True); STATIC=OUT/'static_audit.json'
audit=run_spectral_static_audit(D0,STATIC,device='cuda:0')
print('STATIC OVERALL:',audit.get('decision'),'| STAGE1:',audit.get('stage1_decision'),'| ALTERNATIVES:',audit.get('alternative_controls_decision'))
print('AF2C EQUIVALENCE:',audit.get('af2c_equivalence')); print('RUNTIME TOLERANCE:',audit.get('runtime_tolerance'))
for arm in STAGE1_ARMS:
    e=audit.get('arms',{}).get(arm,{}); f=e.get('failed_gates',[])
    print(f'STATIC {arm}:','PASS' if not f else f'FAIL {f}','| bitwise=',e.get('bitwise_repeat_equal'),'| max_abs_diff=',e.get('max_repeat_difference'))
if audit.get('stage1_decision')!='PASS':
    ignore={'test_accessed','alternative_controls_all_arm_gates_pass','all_arm_gates_pass'}
    common=[k for k,v in audit.get('gates',{}).items() if v is False and k not in ignore]
    failed={a:audit['arms'][a]['failed_gates'] for a in STAGE1_ARMS if audit['arms'][a]['failed_gates']}
    raise RuntimeError(f'STOP: Stage1 static audit gagal. common_failed={common}; arm_failed={failed}')
audit['overall_decision']=audit.get('decision'); audit['scope']='stage1'; audit['scoped_arms']=list(STAGE1_ARMS); audit['decision']='PASS'; audit['training_authorized']=True; audit['test_access_authorized']=False
STATIC.write_text(json.dumps(audit,indent=2)+'\n',encoding='utf-8'); print('STATIC STAGE1 PASS:',STATIC)
# Exercise the exact Stage-1 frontend under PyTorch deterministic mode before spawning training.
previous_det=torch.are_deterministic_algorithms_enabled()
try:
    torch.use_deterministic_algorithms(True,warn_only=False)
    probe=torch.rand(1,3,64,64,device='cuda:0',requires_grad=True)
    for arm in STAGE1_ARMS:
        frontend=SpectralInputEnhancer(frozen_arm_config(arm)).to('cuda:0')
        value=frontend(probe); value.mean().backward(retain_graph=True); probe.grad=None
        print(f'DETERMINISTIC CUDA SMOKE {arm}: PASS')
finally:
    torch.use_deterministic_algorithms(previous_det)
RESUME_ROOT=WORK/'af2-spectral-resume-input'
if RESUME_ROOT.exists(): shutil.rmtree(RESUME_ROOT)
RESUME_ROOT.mkdir(parents=True)
resume_zips=sorted({*INPUT.rglob('af2-spectral-stage1-sequential-output.zip'),*INPUT.rglob('AF2WIN_seed42_output.zip'),*INPUT.rglob('AF2ORI_seed42_output.zip'),*INPUT.rglob('AF2POL_seed42_output.zip'),*INPUT.rglob('AF2SOFT_seed42_output.zip'),*INPUT.rglob('AF2LUM_seed42_output.zip')})
for i,archive in enumerate(resume_zips):
    dst=RESUME_ROOT/f'zip_{i:02d}_{archive.stem}'; dst.mkdir(parents=True)
    with zipfile.ZipFile(archive,'r') as h: h.extractall(dst)
    print('RESUME ZIP EXTRACTED:',archive,'->',dst)
print('RESUME ZIP COUNT:',len(resume_zips))


In [ ]:
ARMS=('AF2WIN','AF2ORI','AF2POL','AF2SOFT','AF2LUM'); SNAPSHOT_BASE=WORK/'af2-spectral-stage1-sequential-output'
def snapshot_stage1():
    a=shutil.make_archive(str(SNAPSHOT_BASE),'zip',OUT); print('SNAPSHOT READY:',a,flush=True); return Path(a)
def restore_exact_arm(arm,config):
    r=restore_spectral_kaggle_run(INPUT,OUT,arm=arm,seed=42,d0_checkpoint=D0,config=config)
    if r is None and any(RESUME_ROOT.iterdir()): r=restore_spectral_kaggle_run(RESUME_ROOT,OUT,arm=arm,seed=42,d0_checkpoint=D0,config=config)
    return r
def run_arm(arm):
    config=REPO/f'configs/af2_spectral/{arm}_yolo26n.yaml'; restored=restore_exact_arm(arm,config)
    result=OUT/'val_reports'/f'{arm}_seed42_result.json'; log=OUT/f'{arm}_seed42_run.log'
    if result.is_file(): print(f'REUSE COMPLETE {arm}: {result}',flush=True); snapshot_stage1(); return
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_spectral_arm','--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(D0),'--static-audit',str(STATIC),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
    print(f'START {arm} | restored={restored}',flush=True)
    with log.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    previous=None
    while process.poll() is None:
        csv=OUT/arm/f'{arm}_seed42'/'results.csv'; epoch=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epoch!=previous: print(f'{arm}: {epoch}/50 epoch | log={log}',flush=True); previous=epoch
        time.sleep(120)
    if process.returncode:
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-180:]) if log.is_file() else '<log tidak ditemukan>'
        raise RuntimeError(f'{arm} gagal: returncode={process.returncode}\n--- {log.name} tail ---\n{tail}')
    if not result.is_file():
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-180:]) if log.is_file() else '<log tidak ditemukan>'
        raise RuntimeError(f'Hasil {arm} tidak ditemukan: {result}\n--- {log.name} tail ---\n{tail}')
    payload=json.loads(result.read_text(encoding='utf-8')); assert payload['evaluation_split']=='val' and payload['test_images_accessed'] is False
    print(f'SELESAI {arm}',flush=True); snapshot_stage1()
for arm in ARMS: run_arm(arm)
print('STAGE 1 COMPLETE:',[str(OUT/'val_reports'/f'{arm}_seed42_result.json') for arm in ARMS])


In [ ]:
rows=[]
for arm in ARMS:
    result=json.loads((OUT/'val_reports'/f'{arm}_seed42_result.json').read_text(encoding='utf-8')); m=result['metrics']
    row={'arm':arm,'Macro':m['macro_map50_95'],'Bottom3':m['bottom3_class_map50_95'],'Worst':m['worst_class_map50_95']}; rows.append(row); print(arm,row)
archive=snapshot_stage1(); print('FINAL STAGE1 ZIP:',archive)
